# Experiment v2 — Scheduled Sampling + 2-layer LSTM

**Thay đổi so với v1:**
- `num_layers=2` trong LSTM (thêm 1 layer)
- **Scheduled sampling**: `teacher_forcing_ratio` giảm dần từ 1.0 → 0.5 qua 10 epoch
- Tận dụng lại toàn bộ features + tokenizer đã cache (không re-extract)

**Mục tiêu:** Train 10 epoch, so sánh BLEU-4 với v1 (baseline: **0.2125**)

In [1]:
# ── Cell 1: Imports & Config ──────────────────────────────────────────────────
import gc
import json
import pickle
import random
import time
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from nltk.translate.bleu_score import (
    corpus_bleu, sentence_bleu, SmoothingFunction,
)
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT   = Path('/Applications/Python_AI/Neural_Image_Caption_Generation')
WORK_DIR       = PROJECT_ROOT / 'workspace'
FEATURES_PATH  = WORK_DIR / 'features_conv_flickr30k.pkl'
TOKENIZER_PATH = WORK_DIR / 'tokenizer_flickr30k.pkl'
SPLITS_PATH    = WORK_DIR / 'flickr30k_splits.json'
DESC_PATH      = WORK_DIR / 'descriptions_flickr30k.txt'
MODEL_DIR      = WORK_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# v2 checkpoint — không ghi đè v1
MODEL_V2_PATH  = MODEL_DIR / 'caption_flickr30k_v2.pt'
HISTORY_V2_PATH = MODEL_DIR / 'training_history_v2.json'

# ── Hyperparams ───────────────────────────────────────────────────────────────
VOCAB_SIZE   = 10_000
MAX_LEN      = 35
EMBED_DIM    = 300
HIDDEN_SIZE  = 512
FEATURE_SIZE = 512
NUM_LAYERS   = 2          # ← tăng từ 1 lên 2
LSTM_DROPOUT = 0.3        # dropout giữa 2 LSTM layers
DROPOUT      = 0.5
BATCH_SIZE   = 64
EPOCHS       = 10         # chỉ train 10 epoch để so sánh nhanh
LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 5.0
LABEL_SMOOTHING = 0.1
BEAM_WIDTH    = 5
FORCE_RETRAIN = True      # luôn train lại cho experiment này

# Scheduled sampling: ratio giảm tuyến tính từ 1.0 → TF_RATIO_END qua EPOCHS
TF_RATIO_START = 1.0
TF_RATIO_END   = 0.5

# ── Device ────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Device      : {device}')
print(f'Epochs      : {EPOCHS}')
print(f'LSTM layers : {NUM_LAYERS}')
print(f'TF ratio    : {TF_RATIO_START} → {TF_RATIO_END} (scheduled sampling)')
print(f'Model v2    : {MODEL_V2_PATH}')

Device      : mps
Epochs      : 10
LSTM layers : 2
TF ratio    : 1.0 → 0.5 (scheduled sampling)
Model v2    : /Applications/Python_AI/Neural_Image_Caption_Generation/workspace/models/caption_flickr30k_v2.pt


/opt/anaconda3/envs/tf-metal/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ── Cell 2: Load existing artifacts ──────────────────────────────────────────
print('Loading features...', end=' ')
with open(FEATURES_PATH, 'rb') as f:
    features_conv = pickle.load(f)
print(f'done — {len(features_conv):,} images')

print('Loading tokenizer...', end=' ')
with open(TOKENIZER_PATH, 'rb') as f:
    tok = pickle.load(f)
word2idx  = tok['word2idx']
idx2word  = tok['idx2word']
vocab_size_actual = tok['vocab_size']
print(f'done — vocab={vocab_size_actual:,}')

print('Loading splits...', end=' ')
with open(SPLITS_PATH) as f:
    splits = json.load(f)
train_ids = set(splits['train'])
val_ids   = set(splits['val'])
test_ids  = set(splits['test'])
print(f'done — train={len(train_ids):,} val={len(val_ids):,} test={len(test_ids):,}')

print('Loading descriptions...', end=' ')
descriptions = {}
with open(DESC_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split(' ', 1)   # format: 'img_id caption'
        if len(parts) != 2:
            continue
        img_id, caption = parts
        descriptions.setdefault(img_id, []).append(caption)
print(f'done — {len(descriptions):,} images')

# Split descriptions
train_desc = {k: v for k, v in descriptions.items() if k in train_ids}
val_desc   = {k: v for k, v in descriptions.items() if k in val_ids}
test_desc  = {k: v for k, v in descriptions.items() if k in test_ids}

# Features by split
train_feat = {k: v for k, v in features_conv.items() if k in train_ids}
val_feat   = {k: v for k, v in features_conv.items() if k in val_ids}
test_feat  = {k: v for k, v in features_conv.items() if k in test_ids}

# Special token IDs
PAD_ID   = 0
UNK_ID   = word2idx.get('<unk>', word2idx.get('unk', 1))
START_ID = word2idx['startseq']
END_ID   = word2idx['endseq']
print(f'Special IDs: PAD={PAD_ID} UNK={UNK_ID} START={START_ID} END={END_ID}')

Loading features... done — 31,783 images
Loading tokenizer... done — vocab=10,000
Loading splits... done — train=29,769 val=1,014 test=1,000
Loading descriptions... done — 31,783 images
Special IDs: PAD=0 UNK=1 START=2 END=3


In [5]:
# ── Cell 3: Dataset & DataLoader ─────────────────────────────────────────────
class CaptionDataset(Dataset):
    def __init__(self, desc: dict, feat: dict, word2idx: dict,
                 max_len: int = MAX_LEN):
        self.samples  = []
        self.word2idx = word2idx
        self.max_len  = max_len
        for img_id, captions in desc.items():
            if img_id not in feat:
                continue
            for cap in captions:
                self.samples.append((img_id, cap))
        self.feat = feat

    def __len__(self):
        return len(self.samples)

    def _encode(self, caption: str):
        tokens = caption.split()
        ids = [self.word2idx.get(w, UNK_ID) for w in tokens]
        # pad / truncate to max_len
        ids = ids[:self.max_len]
        ids += [PAD_ID] * (self.max_len - len(ids))
        return ids

    def __getitem__(self, idx):
        img_id, cap = self.samples[idx]
        feat  = torch.FloatTensor(self.feat[img_id])   # (49, 512)
        ids   = self._encode(cap)                       # [max_len]
        inp   = torch.LongTensor(ids[:-1])              # input  (max_len-1)
        tgt   = torch.LongTensor(ids[1:])               # target (max_len-1)
        return feat, inp, tgt

train_ds = CaptionDataset(train_desc, train_feat, word2idx)
val_ds   = CaptionDataset(val_desc,   val_feat,   word2idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)

print(f'Train samples : {len(train_ds):,}  ({len(train_loader):,} batches)')
print(f'Val samples   : {len(val_ds):,}  ({len(val_loader):,} batches)')

# sanity check
feat_b, inp_b, tgt_b = next(iter(train_loader))
print(f'Batch shapes  : feat={tuple(feat_b.shape)} inp={tuple(inp_b.shape)} tgt={tuple(tgt_b.shape)}')

Train samples : 148,844  (2,326 batches)
Val samples   : 5,070  (80 batches)
Batch shapes  : feat=(64, 49, 512) inp=(64, 34) tgt=(64, 34)


In [6]:
# ── Cell 4: Model — 2-layer LSTM ──────────────────────────────────────────────
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size: int, feature_size: int):
        super().__init__()
        self.W_h = nn.Linear(hidden_size,  hidden_size, bias=False)
        self.W_f = nn.Linear(feature_size, hidden_size, bias=False)
        self.V   = nn.Linear(hidden_size,  1,           bias=False)

    def forward(self, hidden: torch.Tensor, features: torch.Tensor):
        h_exp   = self.W_h(hidden).unsqueeze(1)      # (B, 1, H)
        f_proj  = self.W_f(features)                  # (B, 49, H)
        energy  = self.V(torch.tanh(h_exp + f_proj))  # (B, 49, 1)
        weights = torch.softmax(energy, dim=1)         # (B, 49, 1)
        context = (weights * features).sum(dim=1)      # (B, 512)
        return context, weights.squeeze(-1)


class CaptionDecoder2L(nn.Module):
    """LSTM Decoder với 2 layers + Bahdanau Attention."""

    def __init__(self, vocab_size: int, embed_dim: int, hidden_size: int,
                 feature_size: int = 512, num_layers: int = 2,
                 dropout: float = 0.5, lstm_dropout: float = 0.3,
                 embedding_matrix: Optional[np.ndarray] = None):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_ID)
        if embedding_matrix is not None:
            self.embedding.weight = nn.Parameter(
                torch.FloatTensor(embedding_matrix), requires_grad=True
            )

        self.attention = BahdanauAttention(hidden_size, feature_size)
        self.lstm = nn.LSTM(
            embed_dim + feature_size, hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = lstm_dropout if num_layers > 1 else 0.0,
        )
        self.init_h  = nn.Linear(feature_size, hidden_size)
        self.init_c  = nn.Linear(feature_size, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc_out  = nn.Linear(hidden_size, vocab_size)

    def _init_hidden(self, features: torch.Tensor):
        mean_feat = features.mean(dim=1)                              # (B, 512)
        h = torch.tanh(self.init_h(mean_feat)).unsqueeze(0)          # (1, B, H)
        c = torch.tanh(self.init_c(mean_feat)).unsqueeze(0)          # (1, B, H)
        # repeat cho tất cả layers
        h = h.repeat(self.num_layers, 1, 1)                          # (L, B, H)
        c = c.repeat(self.num_layers, 1, 1)                          # (L, B, H)
        return h, c

    def forward(self, features: torch.Tensor, captions: torch.Tensor):
        """Pure teacher forcing — dùng cho validation/inference."""
        emb  = self.dropout(self.embedding(captions))  # (B, T, E)
        h, c = self._init_hidden(features)
        logits = []
        for t in range(captions.size(1)):
            h_t         = h[-1]                        # last layer (B, H)
            context, _  = self.attention(h_t, features)
            lstm_in     = torch.cat([emb[:, t, :], context], dim=1).unsqueeze(1)
            out, (h, c) = self.lstm(lstm_in, (h, c))
            logit       = self.fc_out(self.dropout(out.squeeze(1)))
            logits.append(logit)
        return torch.stack(logits, dim=1)              # (B, T, V)

    @torch.no_grad()
    def generate_beam(self, features: torch.Tensor, beam_width: int = 5,
                      max_len: int = MAX_LEN) -> str:
        self.eval()
        h0, c0    = self._init_hidden(features)
        beams     = [(0.0, [START_ID], h0, c0)]
        completed = []
        for _ in range(max_len):
            candidates = []
            for score, tokens, bh, bc in beams:
                if tokens[-1] == END_ID:
                    completed.append((score / len(tokens), tokens))
                    continue
                last_t = torch.tensor([[tokens[-1]]], dtype=torch.long,
                                      device=features.device)
                emb    = self.embedding(last_t)        # (1, 1, E)
                ctx, _ = self.attention(bh[-1], features)
                inp    = torch.cat([emb.squeeze(1), ctx], dim=1).unsqueeze(1)
                out, (new_h, new_c) = self.lstm(inp, (bh, bc))
                log_p  = F.log_softmax(self.fc_out(out.squeeze(1)), dim=-1)
                topk   = torch.topk(log_p, beam_width, dim=-1)
                for i in range(beam_width):
                    tok    = topk.indices[0, i].item()
                    new_sc = score + topk.values[0, i].item()
                    candidates.append((new_sc, tokens + [tok], new_h, new_c))
            if not candidates:
                break
            candidates.sort(key=lambda x: x[0] / len(x[1]), reverse=True)
            beams = candidates[:beam_width]
        for score, tokens, bh, bc in beams:
            completed.append((score / len(tokens), tokens))
        if not completed:
            return ''
        _, best_tokens = max(completed, key=lambda x: x[0])
        skip = {START_ID, END_ID, PAD_ID, UNK_ID}
        words = [idx2word.get(t, '') for t in best_tokens
                 if t not in skip and idx2word.get(t, '')]
        return ' '.join(words)


# ── Khởi tạo model ────────────────────────────────────────────────────────────
model = CaptionDecoder2L(
    vocab_size    = vocab_size_actual,
    embed_dim     = EMBED_DIM,
    hidden_size   = HIDDEN_SIZE,
    feature_size  = FEATURE_SIZE,
    num_layers    = NUM_LAYERS,
    dropout       = DROPOUT,
    lstm_dropout  = LSTM_DROPOUT,
).to(device)

total_p    = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model       : CaptionDecoder2L (LSTM-{HIDDEN_SIZE} × {NUM_LAYERS} layers + Bahdanau Attention)')
print(f'Total params: {total_p:,}  (v1 was 11,895,760)')
print(f'Trainable   : {trainable:,}')

Model       : CaptionDecoder2L (LSTM-512 × 2 layers + Bahdanau Attention)
Total params: 13,997,008  (v1 was 11,895,760)
Trainable   : 13,997,008


In [7]:
# ── Cell 5: Training với Scheduled Sampling ───────────────────────────────────
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTHING)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = OneCycleLR(
    optimizer,
    max_lr           = LEARNING_RATE * 3,
    steps_per_epoch  = len(train_loader),
    epochs           = EPOCHS,
    pct_start        = 0.3,
    anneal_strategy  = 'cos',
)


def get_tf_ratio(epoch: int, total: int,
                 start: float = TF_RATIO_START,
                 end: float   = TF_RATIO_END) -> float:
    """Teacher forcing ratio giảm tuyến tính theo epoch."""
    return start - (start - end) * (epoch - 1) / max(total - 1, 1)


def train_epoch_scheduled(loader, tf_ratio: float):
    model.train()
    total_loss = total_correct = total_tokens = total_n = 0
    n_steps  = len(loader)
    log_every = max(1, n_steps // 5)

    for step, (feat, inp, tgt) in enumerate(loader, 1):
        feat = feat.to(device)
        inp  = inp.to(device)
        tgt  = tgt.to(device)
        B, T = tgt.shape

        emb_all = model.dropout(model.embedding(inp))  # (B, T, E)
        h, c    = model._init_hidden(feat)
        logits  = []
        current_emb = emb_all[:, 0, :]                 # token đầu luôn là startseq

        for t in range(T):
            h_t         = h[-1]                        # last layer hidden
            context, _  = model.attention(h_t, feat)
            lstm_in     = torch.cat([current_emb, context], dim=1).unsqueeze(1)
            out, (h, c) = model.lstm(lstm_in, (h, c))
            logit       = model.fc_out(model.dropout(out.squeeze(1)))  # (B, V)
            logits.append(logit)

            if t < T - 1:
                if random.random() < tf_ratio:
                    # Teacher forcing: dùng token thật
                    current_emb = emb_all[:, t + 1, :]
                else:
                    # Scheduled sampling: dùng token model dự đoán
                    pred_token  = logit.detach().argmax(dim=-1)     # (B,)
                    current_emb = model.dropout(model.embedding(pred_token))

        logits = torch.stack(logits, dim=1)            # (B, T, V)
        loss   = criterion(logits.reshape(B * T, -1), tgt.reshape(B * T))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()

        with torch.no_grad():
            pred = logits.argmax(dim=-1)
            mask = tgt != PAD_ID
            total_correct += ((pred == tgt) & mask).sum().item()
            total_tokens  += mask.sum().item()
        total_loss += loss.item() * B
        total_n    += B

        if step % log_every == 0:
            acc = total_correct / max(total_tokens, 1) * 100
            print(f'\r  [{step:>4}/{n_steps}] loss {total_loss/total_n:.4f} | acc {acc:.1f}% | tf={tf_ratio:.2f}',
                  end='', flush=True)

    print()
    return total_loss / max(total_n, 1), total_correct / max(total_tokens, 1) * 100


def val_epoch():
    model.eval()
    total_loss = total_correct = total_tokens = total_n = 0
    with torch.no_grad():
        for feat, inp, tgt in val_loader:
            feat = feat.to(device); inp = inp.to(device); tgt = tgt.to(device)
            B, T = tgt.shape
            logits = model(feat, inp)                   # pure teacher forcing
            loss   = criterion(logits.reshape(B*T, -1), tgt.reshape(B*T))
            pred   = logits.argmax(dim=-1)
            mask   = tgt != PAD_ID
            total_correct += ((pred == tgt) & mask).sum().item()
            total_tokens  += mask.sum().item()
            total_loss    += loss.item() * B
            total_n       += B
    return total_loss / max(total_n, 1), total_correct / max(total_tokens, 1) * 100


# ── Training loop ─────────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
           'tf_ratio': [], 'lr': []}
best_val_loss = float('inf')
SEP = '=' * 70

print(SEP)
print('  TRAINING v2 — Scheduled Sampling + 2-layer LSTM')
print(SEP)
print(f'  {"Epoch":>7} | {"TF":>5} | {"TrainLoss":>9} | {"TrainAcc":>8} | {"ValLoss":>7} | {"ValAcc":>6} | Time')
print(f'  {"-"*68}')

for epoch in range(1, EPOCHS + 1):
    tf_ratio = get_tf_ratio(epoch, EPOCHS)
    t0       = time.time()

    train_loss, train_acc = train_epoch_scheduled(train_loader, tf_ratio)
    val_loss,   val_acc   = val_epoch()
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['tf_ratio'].append(tf_ratio)
    history['lr'].append(optimizer.param_groups[0]['lr'])

    improved = val_loss < best_val_loss
    flag = ' ✅' if improved else ''
    print(f'  Epoch {epoch:02d}/{EPOCHS} | {tf_ratio:.2f} | '
          f'{train_loss:.4f} / {train_acc:.1f}% | '
          f'{val_loss:.4f} / {val_acc:.1f}% | '
          f'{elapsed:.0f}s{flag}')

    if improved:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch, 'model_state': model.state_dict(),
            'val_loss': val_loss, 'num_layers': NUM_LAYERS,
            'optimizer_state': optimizer.state_dict(),
        }, MODEL_V2_PATH)

with open(HISTORY_V2_PATH, 'w') as f:
    json.dump(history, f)

print(SEP)
print(f'  Best val_loss: {best_val_loss:.4f}')
print(f'  Checkpoint  : {MODEL_V2_PATH}')

# ── Plot ─────────────────────────────────────────────────────────────────────
epochs_ran = list(range(1, len(history['train_loss']) + 1))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Training v2 — Scheduled Sampling + 2-layer LSTM', fontweight='bold')

axes[0].plot(epochs_ran, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val',   color='darkorange')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], label='Train', color='steelblue')
axes[1].plot(epochs_ran, history['val_acc'],   label='Val',   color='darkorange')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_ran, history['tf_ratio'], color='green', marker='o', markersize=4)
axes[2].set_title('Teacher Forcing Ratio')
axes[2].set_ylabel('TF Ratio'); axes[2].set_ylim(0, 1.1); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

  TRAINING v2 — Scheduled Sampling + 2-layer LSTM
    Epoch |    TF | TrainLoss | TrainAcc | ValLoss | ValAcc | Time
  --------------------------------------------------------------------
  [2325/2326] loss 6.1412 | acc 18.6% | tf=1.00
  Epoch 01/10 | 1.00 | 6.1410 / 18.6% | 5.1566 / 27.0% | 695s ✅
  [2325/2326] loss 5.0442 | acc 27.8% | tf=0.94
  Epoch 02/10 | 0.94 | 5.0441 / 27.8% | 4.5482 / 32.2% | 699s ✅
  [2325/2326] loss 4.7049 | acc 30.6% | tf=0.89
  Epoch 03/10 | 0.89 | 4.7050 / 30.6% | 4.3022 / 34.6% | 691s ✅
  [2325/2326] loss 4.5872 | acc 31.6% | tf=0.83
  Epoch 04/10 | 0.83 | 4.5871 / 31.6% | 4.1975 / 35.9% | 695s ✅
  [2325/2326] loss 4.5630 | acc 31.6% | tf=0.78
  Epoch 05/10 | 0.78 | 4.5631 / 31.6% | 4.1503 / 36.5% | 696s ✅
  [ 930/2326] loss 4.5764 | acc 31.3% | tf=0.72

KeyboardInterrupt: 

In [ ]:
# ── Cell 6: BLEU Evaluation & So sánh với v1 ─────────────────────────────────
# Load best checkpoint
ckpt = torch.load(MODEL_V2_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded best v2 checkpoint: epoch={ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f}')

# Generate captions cho test set
print(f'Generating captions for {len(test_desc):,} test images (beam_width={BEAM_WIDTH})...')
smooth_fn = SmoothingFunction().method1
actual_refs = []
predicted   = []

with torch.no_grad():
    for img_id, refs in tqdm(test_desc.items(), desc='Beam search'):
        if img_id not in test_feat:
            continue
        feat    = torch.FloatTensor(test_feat[img_id]).unsqueeze(0).to(device)
        caption = model.generate_beam(feat, beam_width=BEAM_WIDTH, max_len=MAX_LEN)
        pred_tokens = caption.split()

        ref_list = []
        for ref in refs:
            toks = [w for w in ref.split() if w not in ('startseq', 'endseq')]
            ref_list.append(toks)

        actual_refs.append(ref_list)
        predicted.append(pred_tokens)

# BLEU scores
bleu1 = corpus_bleu(actual_refs, predicted, weights=(1, 0, 0, 0))
bleu2 = corpus_bleu(actual_refs, predicted, weights=(0.5, 0.5, 0, 0))
bleu3 = corpus_bleu(actual_refs, predicted, weights=(1/3, 1/3, 1/3, 0))
bleu4 = corpus_bleu(actual_refs, predicted, weights=(0.25, 0.25, 0.25, 0.25))

# ── So sánh v1 vs v2 ──────────────────────────────────────────────────────────
V1 = {'BLEU-1': 0.6159, 'BLEU-2': 0.4364, 'BLEU-3': 0.3049, 'BLEU-4': 0.2125}
V2 = {'BLEU-1': bleu1,  'BLEU-2': bleu2,  'BLEU-3': bleu3,  'BLEU-4': bleu4}

print()
print('=' * 60)
print(f'  {"Metric":<10} | {"v1 (30ep TF)":>14} | {"v2 (10ep SS)":>12} | Delta')
print(f'  {"-"*58}')
for metric in ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4']:
    v1_val = V1[metric]
    v2_val = V2[metric]
    delta  = v2_val - v1_val
    sign   = '+' if delta >= 0 else ''
    flag   = '✅' if delta > 0 else ('❌' if delta < -0.005 else '~')
    print(f'  {metric:<10} | {v1_val:>14.4f} | {v2_val:>12.4f} | {sign}{delta:.4f} {flag}')
print('=' * 60)
print(f'  Note: v1 = 30 epochs, v2 = chỉ {EPOCHS} epochs → chưa converge hết')

# ── Sample captions ───────────────────────────────────────────────────────────
print()
print('Sample captions (5 ảnh ngẫu nhiên từ test set):')
sample_ids = random.sample(list(test_desc.keys()), min(5, len(test_desc)))
for img_id in sample_ids:
    feat    = torch.FloatTensor(test_feat[img_id]).unsqueeze(0).to(device)
    caption = model.generate_beam(feat, beam_width=BEAM_WIDTH)
    refs    = [' '.join(w for w in r.split() if w not in ('startseq','endseq'))
               for r in test_desc[img_id][:2]]
    print(f'  [{img_id[:15]}]')
    print(f'    Generated : {caption}')
    print(f'    Ref[0]    : {refs[0]}')
    print(f'    Ref[1]    : {refs[1]}')
    print()